In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =====================================================================
# 1. 初期パラメータ設定 (Experiment Configurations)
# =====================================================================
# [モデル設定]
N = 40                  # 変数の数 (観測の数 P も今回は同じ40)
F = 8.0                 # 強制項
dt = 0.01               # 積分時間ステップ
sampling_dt = 0.05      # 観測・同化の間隔 (6時間相当: 0.25日/5日)
m = 8                   # アンサンブルメンバー数
p = 40                  # 観測点の数 (今回は全点観測なので N と同じ)

# [時間設定]
years = 2
units_per_year = 73.0   # 1年 = 365日 / 5日 = 73ユニット
total_time = years * units_per_year
spin_up_time = units_per_year

steps_total = int(total_time / dt)
steps_spin_up = int(spin_up_time / dt)
sampling_interval = int(sampling_dt / dt)

# [データ同化設定]
H_mat = np.eye(N)                       # 観測演算子 H (全点観測)
R_mat = np.eye(N)                       # 観測誤差共分散 R (対角成分1)

# =====================================================================
# 2. Nature Run (真値) と Observation (観測) の生成
# =====================================================================
def lorenz96(x, F):
    return (np.roll(x, -1, axis=0) - np.roll(x, 2, axis=0)) * np.roll(x, 1, axis=0) - x + F

def M(x_in, dt, steps):
    x_out = x_in.copy()
    for _ in range(steps):
        k1 = lorenz96(x_out, F)
        k2 = lorenz96(x_out + k1 * (dt / 2.0), F)
        k3 = lorenz96(x_out + k2 * (dt / 2.0), F)
        k4 = lorenz96(x_out + k3 * dt, F)
        x_out += (k1 + 2.0*k2 + 2.0*k3 + k4) * (dt / 6.0)
    return x_out

print("Nature Runと観測データを生成中...")
x = np.full(N, F)
x[19] += 1.001

true_states = []
for s in range(steps_total):
    x = M(x, dt, 1)
    if s >= steps_spin_up and (s - steps_spin_up) % sampling_interval == 0:
        true_states.append(x.copy())
true_states = np.array(true_states)

rng_obs = np.random.default_rng(seed=67) 
noise = rng_obs.normal(loc=0.0, scale=1.0, size=true_states.shape)
noise -= np.mean(noise, axis=0)
y_o_data_full = true_states + noise

num_cycles = y_o_data_full.shape[0]

print("生成完了！")

Nature Runと観測データを生成中...
生成完了！


# LETKF + RTPP / RTPS 追加実装

下のセルからが完成版です。  
通常LETKF・RTPP LETKF・RTPS LETKFを同じ関数から呼べるようにしています。

- `method="none"`：通常LETKF
- `method="rtpp"`：RTPP LETKF
- `method="rtps"`：RTPS LETKF

実装上の対応は次の通りです。

$[
\tilde{P}^a =
\left[
\frac{m-1}{\Delta}I
+
(\delta Y^b)^T R^{-1}_{loc}\delta Y^b
\right]^{-1}
]$

$[
w =
\tilde{P}^a
(\delta Y^b)^T R^{-1}_{loc}
d^{o-b}
]$

$[
\bar{x}^a_i =
\bar{x}^b_i + \delta X^b_i w
]$

$[
W =
\left[(m-1)\tilde{P}^a\right]^{1/2}
]$

$[
\delta X^a_i = \delta X^b_i W
]$

RTPP:

$[
\delta X^a_{RTPP,i}
=
(1-\alpha)\delta X^a_i
+
\alpha\sqrt{\Delta}\delta X^b_i
]$

RTPS:

$[
r_i =
1-\alpha
+
\alpha
\frac{\sqrt{\Delta}\sigma_i^b}{\sigma_i^a}
]$

$[
\delta X^a_{RTPS,i}
=
r_i\delta X^a_i
]$


In [ ]:
# =====================================================================
# 3. LETKF + RTPP / RTPS 完成版
# =====================================================================

def symmetric_sqrt(A, eig_floor=1.0e-12):
    """
    対称行列 A の対称平方根 A^{1/2} を返す。
    LETKFでは W W^T = (m-1) P_a_tilde を満たす W を作るために使う。
    """
    A = 0.5 * (A + A.T)
    eigvals, eigvecs = np.linalg.eigh(A)
    eigvals = np.maximum(eigvals, eig_floor)
    return eigvecs @ np.diag(np.sqrt(eigvals)) @ eigvecs.T


def make_H(obs_indices, N):
    """
    観測点 obs_indices に対応する線形観測演算子 H を作る。
    全点観測なら obs_indices = np.arange(N)。
    """
    obs_indices = np.asarray(obs_indices, dtype=int)
    H = np.zeros((len(obs_indices), N))
    for k, j in enumerate(obs_indices):
        H[k, j] = 1.0
    return H


def gaspari_like_weight(distance, sigma):
    """
    元のノートの R_localization_inv に合わせた簡易ガウス型局所化。
    距離がしきい値より大きい観測は重み0。
    """
    cutoff = np.sqrt(10.0 / 3.0) * sigma * 2.0
    if distance < cutoff:
        return np.exp(-(distance**2) / (2.0 * sigma**2))
    return 0.0


def localization_weights_for_grid(i, obs_indices, sigma, N):
    """
    状態変数 i の解析に使う観測局所化重みを返す。
    戻り値 shape: (p_obs,)
    """
    weights = np.zeros(len(obs_indices))
    for k, j in enumerate(obs_indices):
        d = min(abs(i - j), N - abs(i - j))
        weights[k] = gaspari_like_weight(d, sigma)
    return weights


def letkf_analysis(
    X_b,
    y_o,
    H,
    R,
    obs_indices,
    sigma,
    inflation=1.0,
    method="none",
    alpha=0.0,
    eps=1.0e-12,
):
    """
    1同化サイクル分の LETKF 解析を行う。

    method:
        "none" : 通常LETKF
        "rtpp" : RTPP LETKF
        "rtps" : RTPS LETKF

    使用する主な式:

    xb_mean = mean(X_b)
    dX_b    = X_b - xb_mean

    yb_mean = mean(H X_b)
    dY_b    = H X_b - yb_mean
    innov   = y_o - yb_mean

    P_a_tilde =
        [ (m-1)/Delta I + dY_b^T R^{-1}_loc dY_b ]^{-1}

    w =
        P_a_tilde dY_b^T R^{-1}_loc innov

    xa_mean_i =
        xb_mean_i + dX_b_i w

    W =
        [ (m-1) P_a_tilde ]^{1/2}

    dXa_base_i =
        dX_b_i W

    RTPP:
        dXa_i = (1-alpha) dXa_base_i + alpha sqrt(Delta) dX_b_i

    RTPS:
        r_i = 1 - alpha + alpha sqrt(Delta) sigma_b_i / sigma_a_i
        dXa_i = r_i dXa_base_i
    """
    N, m = X_b.shape
    obs_indices = np.asarray(obs_indices, dtype=int)

    # 1. background mean and perturbation
    xb_mean = np.mean(X_b, axis=1)              # shape: (N,)
    dX_b = X_b - xb_mean[:, None]              # shape: (N, m)

    # 2. observation ensemble
    Y_b_ens = H @ X_b                           # shape: (p_obs, m)
    yb_mean = np.mean(Y_b_ens, axis=1)          # shape: (p_obs,)
    dY_b = Y_b_ens - yb_mean[:, None]           # shape: (p_obs, m)

    # 3. innovation
    innovation = y_o - yb_mean                  # shape: (p_obs,)

    # 4. output arrays
    X_a = np.zeros_like(X_b)
    xa_mean = np.zeros(N)
    dXa = np.zeros_like(X_b)

    R_diag = np.diag(R)

    for i in range(N):
        # 5. local R^{-1}
        loc_weights = localization_weights_for_grid(i, obs_indices, sigma, N)
        Rloc_inv_diag = loc_weights / R_diag

        # 6. P_a_tilde
        # P_a_tilde = [ (m-1)/Delta I + dY_b^T Rloc^{-1} dY_b ]^{-1}
        Rinv_dY = Rloc_inv_diag[:, None] * dY_b
        A = ((m - 1) / inflation) * np.eye(m) + dY_b.T @ Rinv_dY
        P_a_tilde = np.linalg.inv(A)
        P_a_tilde = 0.5 * (P_a_tilde + P_a_tilde.T)

        # 7. mean weight
        # w = P_a_tilde dY_b^T Rloc^{-1} innovation
        Rinv_innov = Rloc_inv_diag * innovation
        w = P_a_tilde @ dY_b.T @ Rinv_innov

        # 8. analysis mean
        # xa_mean_i = xb_mean_i + dX_b_i w
        xa_mean[i] = xb_mean[i] + dX_b[i, :] @ w

        # 9. transform matrix
        # W = [ (m-1) P_a_tilde ]^{1/2}
        W = symmetric_sqrt((m - 1) * P_a_tilde)

        # 10. base analysis perturbation
        # dXa_base_i = dX_b_i W
        dXa_base_i = dX_b[i, :] @ W

        # 11. inflation method
        if method == "none":
            dXa_i = dXa_base_i

        elif method == "rtpp":
            # dXa_RTPP = (1-alpha)dXa + alpha sqrt(Delta)dXb
            dXb_inf_i = np.sqrt(inflation) * dX_b[i, :]
            dXa_i = (1.0 - alpha) * dXa_base_i + alpha * dXb_inf_i

        elif method == "rtps":
            # sigma_b_i = std(dXb_i), sigma_a_i = std(dXa_i)
            sigma_b_i = np.std(dX_b[i, :], ddof=1)
            sigma_a_i = np.std(dXa_base_i, ddof=1)

            # r_i = 1-alpha + alpha sqrt(Delta) sigma_b_i / sigma_a_i
            rtps_factor_i = (
                1.0 - alpha
                + alpha * np.sqrt(inflation) * sigma_b_i / (sigma_a_i + eps)
            )
            dXa_i = rtps_factor_i * dXa_base_i

        else:
            raise ValueError("method must be 'none', 'rtpp', or 'rtps'.")

        # 12. perturbation mean should be zero
        dXa_i = dXa_i - np.mean(dXa_i)

        dXa[i, :] = dXa_i
        X_a[i, :] = xa_mean[i] + dXa_i

    return X_a, xa_mean, dXa


def run_letkf_experiment(
    y_o_data,
    true_states,
    obs_indices=None,
    sigma=3.0,
    inflation=1.0,
    method="none",
    alpha=0.0,
    seed=42,
):
    """
    LETKF実験を最初から最後まで回す関数。
    method='none', 'rtpp', 'rtps' を切り替えるだけで比較できる。
    """
    if obs_indices is None:
        obs_indices = np.arange(N)

    obs_indices = np.asarray(obs_indices, dtype=int)
    H = make_H(obs_indices, N)
    R = np.eye(len(obs_indices))

    # 観測データも観測点だけに制限
    y_obs = y_o_data[:, obs_indices]

    num_cycles = y_obs.shape[0]
    record_rmse = np.zeros(num_cycles)
    record_spread = np.zeros(num_cycles)

    # 初期アンサンブル
    rng_enkf = np.random.default_rng(seed=seed)
    init_noise = rng_enkf.normal(0.0, 0.1, size=(N, m))
    init_noise -= np.mean(init_noise, axis=1, keepdims=True)

    x_raw_init = np.full(N, F)
    x_raw_init[min(19, N - 1)] += 1.001
    X_a = M(x_raw_init[:, None] + init_noise, dt, steps_spin_up)

    for t in range(num_cycles):
        # forecast
        X_b = M(X_a, dt, sampling_interval)

        # analysis
        X_a, xa_mean, dXa = letkf_analysis(
            X_b=X_b,
            y_o=y_obs[t],
            H=H,
            R=R,
            obs_indices=obs_indices,
            sigma=sigma,
            inflation=inflation,
            method=method,
            alpha=alpha,
        )

        # diagnostics
        record_rmse[t] = np.sqrt(np.mean((xa_mean - true_states[t]) ** 2))
        record_spread[t] = np.mean(np.std(X_a, axis=1, ddof=1))

    return record_rmse, record_spread


In [ ]:
# =====================================================================
# 4. まずは短い期間で動作確認
# =====================================================================
# 全1460サイクルを3手法で回すと少し時間がかかるので、
# まずは先頭200サイクルだけで確認する。

num_eval_cycles = 200

sigma = 3.0
inflation = 1.05
alpha = 0.7
obs_indices = np.arange(N)   # 全点観測

results = {}

for method, a in [
    ("none", 0.0),
    ("rtpp", alpha),
    ("rtps", alpha),
]:
    print(f"Running {method} ...")
    rmse, spread = run_letkf_experiment(
        y_o_data=y_o_data_full[:num_eval_cycles],
        true_states=true_states[:num_eval_cycles],
        obs_indices=obs_indices,
        sigma=sigma,
        inflation=inflation,
        method=method,
        alpha=a,
        seed=42,
    )
    results[method] = {"rmse": rmse, "spread": spread}
    print(f"{method}: mean RMSE = {np.mean(rmse):.4f}, mean spread = {np.mean(spread):.4f}")

print("Done.")


In [ ]:
# =====================================================================
# 5. RMSEとSpreadの比較プロット
# =====================================================================
plt.figure(figsize=(10, 4))
for method in results:
    plt.plot(results[method]["rmse"], label=method)
plt.xlabel("Assimilation cycle")
plt.ylabel("RMSE")
plt.title("RMSE comparison")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 4))
for method in results:
    plt.plot(results[method]["spread"], label=method)
plt.xlabel("Assimilation cycle")
plt.ylabel("Spread")
plt.title("Spread comparison")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# =====================================================================
# 6. 全期間で実験したい場合
# =====================================================================
# 時間がかかる場合があるので、必要になったら実行する。

# num_eval_cycles = num_cycles
# results_full = {}
#
# for method, a in [
#     ("none", 0.0),
#     ("rtpp", alpha),
#     ("rtps", alpha),
# ]:
#     print(f"Running {method} full experiment ...")
#     rmse, spread = run_letkf_experiment(
#         y_o_data=y_o_data_full[:num_eval_cycles],
#         true_states=true_states[:num_eval_cycles],
#         obs_indices=obs_indices,
#         sigma=sigma,
#         inflation=inflation,
#         method=method,
#         alpha=a,
#         seed=42,
#     )
#     results_full[method] = {"rmse": rmse, "spread": spread}
#     print(f"{method}: mean RMSE = {np.mean(rmse):.4f}, mean spread = {np.mean(spread):.4f}")


# 実装メモ

元の途中コードから特に直した点です。

1. `X_b_mean = np.mean(X_b, axis=1, keepdims=True)` とした場合、`dX_b = X_b - X_b_mean[:, None]` にすると次元が増えてしまいます。  
   完成版では `xb_mean` を shape `(N,)` にして、`dX_b = X_b - xb_mean[:, None]` に統一しました。

2. 解析平均と解析摂動を分けました。  
   先に `xa_mean[i] = xb_mean[i] + dX_b[i, :] @ w` を作り、その後に `dXa_i` を足して `X_a[i, :]` を作ります。

3. RTPPは
   `dXa_i = (1-alpha) * dXa_base_i + alpha * sqrt(inflation) * dX_b[i, :]`
   として、講義資料の式にそのまま対応させました。

4. RTPSは
   `sigma_b_i = std(dX_b[i, :])` と `sigma_a_i = std(dXa_base_i)` を使い、
   `rtps_factor_i` を変数ごとに計算する形にしました。

5. `method` を切り替えるだけで、通常LETKF・RTPP・RTPSを比較できるようにしました。
